# 10. 피처 파이프라인 조립 랩

## 연구 질문

> 피처셋 config가 만드는 조립 순서와 fit 경계가 **train/test 대칭**을 보장하는가?

09번 노트북은 피처셋을 config로 표현해도 재현성 계약이 유지된다는 것을 닫았다.
그러나 config가 옳다고 해서 그것이 만드는 feature matrix가 옳은 것은 아니다.
조립 단계의 순서가 한 칸만 어긋나도 컬럼 집합이 달라지고, fit 범위가 한 번만 새면
2024 holdout 점수가 낙관적으로 부풀어 오른다.

이 노트북은 조립기 `build_feature_pipeline`이 **무엇을 어떤 순서로 하고, 어디까지만
학습 데이터를 보는지**를 공식 데이터로 확인한다.

점수를 재는 노트북이 아니다. 점수 재현은 11번 노트북이 담당한다.

## 목차

1. 조립은 왜 배선이 아니라 순서 계약인가 (Decision Box ⑥)
2. 계약대로 조립됐는지 컬럼으로 확인한다
3. fit 경계 — train만 보고 test는 변환만 받는다 (Decision Box ⑦)
4. 터빈 좌표 로더 — 노트북 셀에서 `src`로 (Decision Box ⑧)
5. own-group superset — 한 번 조립하고 target별로 자른다 (Decision Box ⑨⑩)
6. 종합 결론

## 이 노트북의 전제

- 공식 원자료는 Git에 커밋하지 않으므로 로컬 `data/raw/open/`에서 읽는다.
- 모델 파일, 제출 CSV, registry 행은 만들지 않는다. 조립 계약 검증만 수행한다.
- 설계 근거는 `docs/design/05-feature-set-promotion.md` v1.0의 4.5·4.6절을 따른다.

In [1]:
from pathlib import Path
import re
import sys
import time

import pandas as pd


def resolveProjectRoot():
  """worktree/일반 체크아웃 어디서 실행하든 src와 원자료를 찾는다."""
  here = Path.cwd().resolve()
  for candidate in [here, *here.parents]:
    if (candidate / "src" / "baram").is_dir():
      return candidate
  raise RuntimeError("src/baram을 찾지 못했습니다")


def resolveOfficialDataDir(projectRoot):
  for candidate in [projectRoot, *projectRoot.parents]:
    dataDir = candidate / "data" / "raw" / "open"
    if (dataDir / "train" / "train_labels.csv").is_file():
      return dataDir
  raise RuntimeError("공식 데이터 디렉터리를 찾지 못했습니다")


projectRoot = resolveProjectRoot()
if str(projectRoot / "src") not in sys.path:
  sys.path.insert(0, str(projectRoot / "src"))
dataDir = resolveOfficialDataDir(projectRoot)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print(f"project root : {projectRoot.name}")
print(f"official data: {dataDir}")

project root : codex-to-claude-transition-a9327d
official data: C:\Users\kik32\workspace\Dacon\baram-2026-wind-power-forecasting\data\raw\open


09번과 같은 탐색 규약이다. 데이터를 찾지 못하면 여기서 멈추므로, 원자료 없이 만들어진
결론이 노트북에 남는 일은 없다.

---

## 1. 조립은 왜 배선이 아니라 순서 계약인가

`build_feature_pipeline`이 하는 일은 네 단계다. 단계 자체는 단순하지만 **순서가 계약**이다.

In [2]:
assemblyOrder = pd.DataFrame(
  [
    ("①", "derive_wind_vector_features", "raw grid 행", "speed·wind-from sin/cos 파생", "config.wind_vector"),
    ("②", "build_weather_features", "raw grid 행", "forecast 시각 단위 통계 집계", "config.statistics / include_lead"),
    ("③", "TurbineSpatialPooler.transform", "①이 끝난 raw 행", "터빈 그룹별 공간 pooling", "config.spatial"),
    ("④", "calendar_features", "forecast 시각", "달력 피처를 맨 앞에 결합", "config.calendar"),
  ],
  columns=["단계", "함수", "입력", "하는 일", "켜는 조건"],
)
assemblyOrder

,단계,함수,입력,하는 일,켜는 조건
0,①,derive_wind_vector_features,raw grid 행,speed·wind-from sin/cos 파생,config.wind_vector
1,②,build_weather_features,raw grid 행,forecast 시각 단위 통계 집계,config.statistics / include_lead
2,③,TurbineSpatialPooler.transform,①이 끝난 raw 행,터빈 그룹별 공간 pooling,config.spatial
3,④,calendar_features,forecast 시각,달력 피처를 맨 앞에 결합,config.calendar


### Decision Box ⑥ — 공간 pooling의 입력은 집계 결과가 아니라 ①이 끝난 raw 행이다

**선택지**

- (A) ② 집계가 끝난 forecast 시각 단위 프레임을 pooling한다
- (B) ① wind vector 파생이 끝난 raw grid 행을 pooling한다

**근거**

(A)는 직관적으로 보인다. 이미 시각 단위로 줄어든 프레임을 다루니 계산량도 적다.
그러나 집계는 grid 축을 이미 뭉갠 뒤다. 격자별 값이 사라진 프레임에 터빈 거리 가중치를
곱하면 **가중할 대상이 없다.** 공간 pooling의 정의 자체가 성립하지 않는다.

(B)는 격자 축이 살아 있는 상태에서 터빈-격자 거리로 가중 평균을 만든다. 이때 ①이 먼저
끝나 있어야 `windvec_*_speed`, `windvec_*_from_sin/cos` 같은 파생 변수도 pooling 대상에
포함된다. 노트북 08이 550개를 만든 것이 바로 이 순서였다.

**채택: (B)** — 그리고 순서를 바꾸면 무엇이 달라지는지 2절에서 숫자로 확인한다.

---

## 2. 계약대로 조립됐는지 컬럼으로 확인한다

먼저 공식 데이터와 터빈 좌표를 읽는다.

In [3]:
from baram.feature_config import FeatureSetConfig, SpatialPoolingConfig, FEATURE_SETS, get_feature_set
from baram.feature_pipeline import build_feature_pipeline
from baram.features.turbine_metadata import load_turbine_locations
from baram.features.weather_grid import SPATIAL_STATISTICS


def readOfficial(relativePath):
  return pd.read_csv(dataDir / relativePath, encoding="utf-8-sig")


trainLabels = readOfficial("train/train_labels.csv")
trainLabels["kst_dtm"] = pd.to_datetime(trainLabels["kst_dtm"])
ldapsTrain = readOfficial("train/ldaps_train.csv")
gfsTrain = readOfficial("train/gfs_train.csv")
ldapsTest = readOfficial("test/ldaps_test.csv")
gfsTest = readOfficial("test/gfs_test.csv")
sampleSubmission = readOfficial("sample_submission.csv")

trainTimeIndex = trainLabels["kst_dtm"]
testTimeIndex = pd.to_datetime(sampleSubmission["forecast_kst_dtm"])
turbines = load_turbine_locations(dataDir / "info.xlsx")

pd.DataFrame(
  [
    ("train_labels", *trainLabels.shape),
    ("ldaps_train", *ldapsTrain.shape),
    ("gfs_train", *gfsTrain.shape),
    ("ldaps_test", *ldapsTest.shape),
    ("gfs_test", *gfsTest.shape),
    ("sample_submission", *sampleSubmission.shape),
    ("info.xlsx 터빈", *turbines.shape),
  ],
  columns=["파일", "rows", "cols"],
)

,파일,rows,cols
0,train_labels,26304,4
1,ldaps_train,420864,35
2,gfs_train,236736,40
3,ldaps_test,140160,35
4,gfs_test,78840,40
5,sample_submission,8760,5
6,info.xlsx 터빈,17,5


행 수는 00번 감사 노트북이 잠근 값과 같다. 학습 시간축 26,304행(2022~2024 3년),
제출 시간축 8,760행(2025년 1년)이다.

전체를 조립하기 전에 **작은 표본으로 먼저** 계약이 성립하는지 본다.
2022년 1월을 train, 2월을 test로 삼아 가장 복잡한 프리셋을 통과시킨다.
여기서 깨지면 전체를 돌릴 이유가 없다.

In [4]:
def monthMask(frame, column, month):
  return pd.to_datetime(frame[column]).dt.strftime("%Y-%m") == month


pilotTrainMonth, pilotTestMonth = "2022-01", "2022-02"
pilotTrainIndex = trainLabels.loc[monthMask(trainLabels, "kst_dtm", pilotTrainMonth), "kst_dtm"]
pilotTestIndex = trainLabels.loc[monthMask(trainLabels, "kst_dtm", pilotTestMonth), "kst_dtm"]

pilotPipeline = build_feature_pipeline(
  get_feature_set("spatial_idw2_all_group"),
  train_time_index=pilotTrainIndex,
  test_time_index=pilotTestIndex,
  train_ldaps=ldapsTrain[monthMask(ldapsTrain, "forecast_kst_dtm", pilotTrainMonth)],
  train_gfs=gfsTrain[monthMask(gfsTrain, "forecast_kst_dtm", pilotTrainMonth)],
  test_ldaps=ldapsTrain[monthMask(ldapsTrain, "forecast_kst_dtm", pilotTestMonth)],
  test_gfs=gfsTrain[monthMask(gfsTrain, "forecast_kst_dtm", pilotTestMonth)],
  turbine_locations=turbines,
)

pd.DataFrame(
  [
    ("파일럿 train", pilotTrainMonth, *pilotPipeline.train_matrix.shape),
    ("파일럿 test", pilotTestMonth, *pilotPipeline.test_matrix.shape),
  ],
  columns=["구분", "기간", "rows", "features"],
).assign(
  컬럼_대칭=list(pilotPipeline.train_matrix.columns) == list(pilotPipeline.test_matrix.columns),
)

,구분,기간,rows,features,컬럼_대칭
0,파일럿 train,2022-01,743,550,True
1,파일럿 test,2022-02,672,550,True


743행과 672행에 대해 컬럼 550개가 대칭으로 나왔다.

1월이 744행(31일 × 24시간)이 아니라 743행인 데는 이유가 있다. 공식 `train_labels`는
2022-01-01 **01:00**에서 시작하고, 각 날의 00:00을 전날 마지막 시각으로 셈한다.
그래서 연도별 행 수가 2022년 8,759 / 2023년 8,760 / 2024년 8,784(윤년) / **2025년 1행**이 되고,
마지막 행은 2025-01-01 00:00이다. 이 경계 행 1건은 2024 holdout을 자를 때 반드시
빠져야 하는 값이며, 11번 노트북에서 다시 다룬다.

표본 크기가 1/36로 줄어도 **컬럼 계약은 행 수와 무관하게 성립한다.** 조립기가 데이터
분포가 아니라 스키마에서 컬럼을 결정한다는 뜻이고, 이것이 train/test 대칭의 전제다.

이제 프리셋 4종을 전체 데이터로 조립한다.

In [5]:
pipelines = {}
timings = []
for name in FEATURE_SETS:
  config = get_feature_set(name)
  started = time.perf_counter()
  pipelines[name] = build_feature_pipeline(
    config,
    train_time_index=trainTimeIndex,
    test_time_index=testTimeIndex,
    train_ldaps=ldapsTrain,
    train_gfs=gfsTrain,
    test_ldaps=ldapsTest,
    test_gfs=gfsTest,
    turbine_locations=turbines if config.uses_spatial else None,
  )
  elapsed = time.perf_counter() - started
  profile = pipelines[name].weather_profile
  timings.append(
    (
      name,
      profile["aggregated_feature_count"],
      profile["spatial_feature_count"],
      profile["total_feature_count"],
      max(len(columns) for columns in pipelines[name].target_feature_columns.values()),
      min(len(columns) for columns in pipelines[name].target_feature_columns.values()),
      round(elapsed, 1),
    )
  )

pd.DataFrame(
  timings,
  columns=["preset", "집계", "pooling", "superset", "target 최대", "target 최소", "조립 초"],
)

,preset,집계,pooling,superset,target 최대,target 최소,조립 초
0,official_mean,67,0,76,76,76,0.9
1,expanded_vector,310,0,319,319,319,1.8
2,spatial_idw2_all_group,310,231,550,550,550,10.4
3,spatial_nearest_own_group,310,231,550,396,396,11.1


### 2-1. 프리셋별 피처 수가 랩 노트북과 정확히 일치한다

| preset | superset | 랩 노트북 | 대조 |
|--------|---------:|----------:|------|
| `official_mean` | 76 | 74 (06~08 통제군) | lead 2개 차이 — Decision Box ①의 결과 |
| `expanded_vector` | 319 | 319 (07) | 일치 |
| `spatial_idw2_all_group` | 550 | 550 (08) | 일치 |
| `spatial_nearest_own_group` | 550 → target 396 | 396 (08) | 일치 |

주목할 것은 마지막 행이다. own-group 프리셋도 **superset은 550**이고 target별로 396을
쓴다. 파이프라인은 한 번만 돌고, 슬라이스는 학습 직전에 일어난다. 설계서 2.4절의 결정이다.

조립 시간은 공간 프리셋도 12초 안쪽이다. RandomForest 학습이 이보다 훨씬 오래 걸리므로
조립 비용은 실험 반복의 병목이 아니다.

이제 컬럼이 계약된 순서대로 놓였는지 본다.

In [6]:
POOLED_PATTERN = re.compile(r"_group_\d+_(nearest|idw)$")


def segmentOf(name):
  """컬럼 이름 하나를 계약 구획으로 분류한다."""
  if not (name.startswith("ldaps_") or name.startswith("gfs_")):
    return "calendar"
  source = "ldaps" if name.startswith("ldaps_") else "gfs"
  return f"{source} {'pooling' if POOLED_PATTERN.search(name) else '집계'}"


idwColumns = list(pipelines["spatial_idw2_all_group"].feature_columns)
segments = [segmentOf(name) for name in idwColumns]

boundaries = []
for position, (name, segment) in enumerate(zip(idwColumns, segments)):
  if position == 0 or segment != segments[position - 1]:
    boundaries.append((len(boundaries) + 1, segment, position, name))

pd.DataFrame(boundaries, columns=["순번", "구획", "시작 위치", "첫 컬럼"]).assign(
  컬럼_수=[segments.count(segment) for _, segment, _, _ in boundaries],
)

,순번,구획,시작 위치,첫 컬럼,컬럼_수
0,1,calendar,0,month,9
1,2,ldaps 집계,9,ldaps_heightAboveGround_10_10u_mean,133
2,3,gfs 집계,142,gfs_heightAboveGround_10_10u_mean,177
3,4,ldaps pooling,319,ldaps_heightAboveGround_10_10u_group_1_idw,99
4,5,gfs pooling,418,gfs_heightAboveGround_10_10u_group_1_idw,132


### 2-2. 다섯 구획이 계약 순서대로 한 덩어리씩 놓였다

`calendar(9) → ldaps 집계(133) → gfs 집계(177) → ldaps pooling(99) → gfs pooling(132)`
순으로 9 + 133 + 177 + 99 + 132 = 550이다. 각 구획이 **흩어지지 않고 연속 블록**을 이룬다.

pooling 컬럼 231개는 raw 변수 77개(ldaps 33 + gfs 44) × 3그룹이다.
`ldaps pooling` 99 = 33 × 3, `gfs pooling` 132 = 44 × 3으로 나뉜다.

이 순서가 왜 중요한가. `max_features="sqrt"`인 RandomForest는 분기마다 컬럼을 무작위로
고르는데, 그 무작위 추출은 **컬럼 순서에 의존한다.** 이름이 같아도 순서가 다르면 다른
트리가 자라고 점수가 달라진다. 순서를 계약으로 못 박고 테스트로 잠근 이유다.

이제 Decision Box ⑥을 숫자로 검증한다. ①을 끄면 ③의 대상이 줄어드는가?

In [7]:
noVectorIdw = FeatureSetConfig(
  name="probe_no_vector_idw",
  statistics=SPATIAL_STATISTICS,
  include_lead=True,
  wind_vector=False,
  spatial=SpatialPoolingConfig(methods=("idw",), idw_power=2.0, scope="all_group"),
)
noVectorPipeline = build_feature_pipeline(
  noVectorIdw,
  train_time_index=trainTimeIndex,
  test_time_index=testTimeIndex,
  train_ldaps=ldapsTrain,
  train_gfs=gfsTrain,
  test_ldaps=ldapsTest,
  test_gfs=gfsTest,
  turbine_locations=turbines,
)


def poolingProfile(columns):
  pooled = [name for name in columns if POOLED_PATTERN.search(name)]
  rawVariables = sorted({POOLED_PATTERN.sub("", name) for name in pooled})
  return len(columns), len(pooled), len(rawVariables), rawVariables


withVector = poolingProfile(idwColumns)
withoutVector = poolingProfile(list(noVectorPipeline.feature_columns))
lostVariables = sorted(set(withVector[3]) - set(withoutVector[3]))

print(f"① 켬  : 총 {withVector[0]} / pooling {withVector[1]} / pooling 대상 raw 변수 {withVector[2]}")
print(f"① 끔  : 총 {withoutVector[0]} / pooling {withoutVector[1]} / pooling 대상 raw 변수 {withoutVector[2]}")
print(f"사라진 raw 변수 {len(lostVariables)}개 -> pooling 컬럼 {len(lostVariables) * 3}개 감소")
print()
for name in lostVariables:
  print("  -", name)

① 켬  : 총 550 / pooling 231 / pooling 대상 raw 변수 77
① 끔  : 총 466 / pooling 195 / pooling 대상 raw 변수 65
사라진 raw 변수 12개 -> pooling 컬럼 36개 감소

  - gfs_windvec_100m_from_cos
  - gfs_windvec_100m_from_sin
  - gfs_windvec_100m_speed
  - gfs_windvec_10m_from_cos
  - gfs_windvec_10m_from_sin
  - gfs_windvec_10m_speed
  - gfs_windvec_80m_from_cos
  - gfs_windvec_80m_from_sin
  - gfs_windvec_80m_speed
  - ldaps_windvec_10m_from_cos
  - ldaps_windvec_10m_from_sin
  - ldaps_windvec_10m_speed


### 2-3. ①을 끄면 ③이 pooling할 변수 12개가 통째로 사라진다

wind vector 파생을 끄자 pooling 대상 raw 변수가 77 → 65로 줄고, 최종 피처가
550 → 466으로 84개 줄었다. 사라진 12개는 전부 `windvec_*` 파생 변수이고,
12 × 3그룹 = 36개의 pooling 컬럼과 48개의 집계 컬럼이 함께 없어졌다.

이것이 "순서가 계약"이라는 말의 실체다. ①과 ③은 독립된 스위치가 아니라 **①의 출력이
③의 입력**이다. 만약 ③을 ② 뒤에 놓았다면 wind vector를 켜든 끄든 pooling 대상은
집계 컬럼이 되어 노트북 08의 550개는 영영 재현되지 않는다.

---

## 3. fit 경계 — train만 보고 test는 변환만 받는다

조립 순서가 맞아도 fit 범위가 새면 검증 점수가 거짓말을 한다.
이 위험은 문헌에서 반복해 지적돼 왔다.

### Decision Box ⑦ — 공간 pooler는 train grid geometry에만 fit하고 test는 transform만 받는다

**선택지**

- (A) train+test 전체 grid로 pooler를 fit한다 (좌표는 label이 아니므로 안전해 보인다)
- (B) train grid geometry에만 fit하고, test에는 transform만 적용한다

**근거**

(A)의 유혹은 "격자 좌표는 정답이 아니니 봐도 된다"는 데서 온다. 그러나 Moscovich와
Rosset은 **비지도(unsupervised) 전처리조차** resampling loop 밖에서 파라미터를 추정하면
교차검증 추정치에 상당한 편향을 만든다는 것을 보였다
(*On the cross-validation bias due to unsupervised pre-processing*, JRSS-B 84(4), 2022).
Lones 역시 시계열에서는 look-ahead bias가 별도의 누수 경로가 된다고 정리한다
(*How to avoid machine learning pitfalls*, arXiv:2108.02497).

우리 경우 pooler가 fit하는 것은 격자 좌표와 거리 가중치다. 2025년 test 격자가 2022~2024와
다르다면, (A)는 그 차이를 조용히 흡수해 **holdout이 미래를 미리 본 상태**가 된다.
(B)는 차이가 있으면 흡수하지 않고 예외로 드러낸다.

**채택: (B)** — 그리고 drift가 실제로 검출되는지 아래에서 확인한다.

median imputer도 같은 규칙을 따른다. `train_random_forest_baseline`은 imputer를
train feature matrix에만 fit하고 test는 `transform`만 받는다.

In [8]:
symmetry = []
for name, pipeline in pipelines.items():
  trainColumns = list(pipeline.train_matrix.columns)
  testColumns = list(pipeline.test_matrix.columns)
  symmetry.append(
    (
      name,
      len(trainColumns),
      len(testColumns),
      trainColumns == testColumns,
      pipeline.train_matrix.shape[0],
      pipeline.test_matrix.shape[0],
    )
  )

symmetryFrame = pd.DataFrame(
  symmetry,
  columns=["preset", "train 컬럼", "test 컬럼", "이름·순서 동일", "train 행", "test 행"],
)
print("입력 프레임이 조립 과정에서 변형되지 않았는가")
print("  ldaps_train :", ldapsTrain.shape, "| gfs_test :", gfsTest.shape)
print("  원본 컬럼 유지 :", "windvec_10m_speed" not in ldapsTrain.columns)
symmetryFrame

입력 프레임이 조립 과정에서 변형되지 않았는가
  ldaps_train : (420864, 35) | gfs_test : (78840, 40)
  원본 컬럼 유지 : True


,preset,train 컬럼,test 컬럼,이름·순서 동일,train 행,test 행
0,official_mean,76,76,True,26304,8760
1,expanded_vector,319,319,True,26304,8760
2,spatial_idw2_all_group,550,550,True,26304,8760
3,spatial_nearest_own_group,550,550,True,26304,8760


### 3-1. 네 프리셋 모두 train/test 컬럼이 이름·순서까지 동일하다

`build_feature_pipeline`은 마지막에 `list(train_matrix.columns) != list(test_matrix.columns)`를
직접 검사해 다르면 예외를 던진다. 대칭은 결과가 아니라 **강제된 사후 조건**이다.

입력 프레임도 그대로다. ① wind vector 파생은 원본 `ldaps_train`에 컬럼을 추가하지 않고
새 프레임을 만든다. 조립기를 두 번 호출해도 같은 결과가 나온다는 뜻이고, 노트북에서
셀을 재실행할 때 조용히 값이 달라지는 사고를 막는다.

이제 fit 경계가 실제로 감시되는지 본다. test 격자 좌표를 일부러 흔들어 본다.

In [9]:
from baram.features.wind_vector import derive_wind_vector_features

ldapsPooler = pipelines["spatial_idw2_all_group"].spatial_poolers["ldaps"]
ldapsTestVector = derive_wind_vector_features(ldapsTest, source="ldaps")

driftChecks = []

# 정상 경로: fit 당시와 같은 geometry면 통과한다
normalOutput = ldapsPooler.transform(ldapsTestVector)
driftChecks.append(("정상 test 프레임", "통과", f"{normalOutput.shape[0]}행 × {normalOutput.shape[1]}열"))

# 좌표 drift: 한 격자의 위도를 통째로 옮긴다
movedGrid = ldapsTestVector.copy()
firstGrid = movedGrid["grid_id"].iloc[0]
movedGrid.loc[movedGrid["grid_id"] == firstGrid, "latitude"] += 0.5
try:
  ldapsPooler.transform(movedGrid)
  driftChecks.append(("격자 좌표 이동", "검출 실패", "-"))
except ValueError as error:
  driftChecks.append(("격자 좌표 이동", "차단", str(error)[:60]))

# 격자 소멸: 격자 하나를 빼 본다
droppedGrid = ldapsTestVector[ldapsTestVector["grid_id"] != firstGrid]
try:
  ldapsPooler.transform(droppedGrid)
  driftChecks.append(("격자 1개 소멸", "검출 실패", "-"))
except ValueError as error:
  driftChecks.append(("격자 1개 소멸", "차단", str(error)[:60]))

# 스키마 drift: wind vector 파생을 건너뛴 raw 프레임
try:
  ldapsPooler.transform(ldapsTest)
  driftChecks.append(("① 생략한 raw 프레임", "검출 실패", "-"))
except ValueError as error:
  driftChecks.append(("① 생략한 raw 프레임", "차단", str(error)[:60]))

pd.DataFrame(driftChecks, columns=["시나리오", "결과", "메시지/형태"])

,시나리오,결과,메시지/형태
0,정상 test 프레임,통과,8760행 × 100열
1,격자 좌표 이동,차단,fit/transform grid 좌표 drift
2,격자 1개 소멸,차단,ldaps_weather grid 수 계약 위반: forecast당 정확히 16개가...
3,① 생략한 raw 프레임,차단,fit/transform 컬럼 이름 또는 순서 drift: fit=['forecas...


### 3-2. geometry가 흔들리면 조용히 흡수하지 않고 즉시 멈춘다

세 가지 drift가 모두 `ValueError`로 차단됐다. 특히 마지막 항목이 Decision Box ⑥과 이어진다.
①을 건너뛴 raw 프레임을 pooler에 넣으면 **컬럼 스키마 자체가 다르므로** transform 단계에서
막힌다. 조립 순서 위반이 런타임에 드러나는 두 번째 방어선이다.

여기서 한 가지를 분명히 해 둔다. 이 검사는 "2025년 격자가 2022~2024와 같다"를 확인해
줄 뿐, **결측을 메워 주지는 않는다.** 설계서 R5가 기록한 대로 2025년 test에는 LDAPS
forecast-variable 쌍 47개가 전 격자 결측이고, 이 계약은 그것을 NaN으로 보존한 뒤
train에서 fit한 median으로만 채운다. 공간 pooling으로 없는 관측을 복원하지 않는다.

---

## 4. 터빈 좌표 로더 — 노트북 셀에서 `src`로

공간 프리셋은 터빈 17기의 좌표 없이는 한 줄도 실행되지 않는다.
그런데 이 변환은 승격 전까지 노트북 08 셀 안에만 있었다.

### Decision Box ⑧ — 좌표 검증을 호출부가 아니라 로더 안에 둔다

**선택지**

- (A) 로더는 파싱만 하고, 터빈 수·그룹 구성 검증은 쓰는 쪽에서 한다
- (B) 로더가 파싱과 동시에 공식값 대조까지 마치고, 어긋나면 로드 자체를 실패시킨다

**근거**

`info.xlsx`는 공식 배포 파일이지만 그룹 열이 **그룹의 첫 터빈에만** 채워져 있어 전방 채움이
필요하다. (A)를 고르면 전방 채움이 한 칸 밀렸을 때 group_3에 6기가 잡혀도 파이프라인은
그대로 돌아가고, 잘못된 가중치로 만들어진 550개 피처가 조용히 학습에 들어간다.

(B)는 터빈 수 17, 그룹별 6·6·5, 그룹 용량 21.6·21.6·21.0 MW를 로드 시점에 대조한다.
DMS 좌표 형식도 정규식으로 강제한다. 잘못된 좌표로 만든 모델은 되돌릴 수 없지만,
로드 실패는 즉시 눈에 띈다.

**채택: (B)** — 검증 실패 비용이 조용한 오염 비용보다 훨씬 싸다.

In [10]:
groupSummary = (
  turbines.groupby("group_id")
  .agg(
    터빈수=("turbine_id", "count"),
    설비용량MW=("capacity_mw", "sum"),
    위도최소=("latitude", "min"),
    위도최대=("latitude", "max"),
    경도최소=("longitude", "min"),
    경도최대=("longitude", "max"),
  )
  .round(6)
)
print("터빈 총 수 :", len(turbines))
print("제작사 구성 :", turbines["turbine_id"].str.split("-").str[0].value_counts().to_dict())
groupSummary

터빈 총 수 : 17
제작사 구성 : {'VESTAS': 12, 'UNISON': 5}


,터빈수,설비용량MW,위도최소,위도최대,경도최소,경도최대
group_id,,,,,,
group_1,6,21.6,37.282114,37.291167,128.949542,128.956933
group_2,6,21.6,37.275161,37.287833,128.959631,128.967828
group_3,5,21.0,37.268564,37.283258,128.962492,128.976578


공식값과 일치한다. group_1·group_2는 VESTAS 6기씩 3.6 MW, group_3은 UNISON 5기 4.2 MW로
각각 21.6 / 21.6 / 21.0 MW다. 17기가 위도 37.268~37.291, 경도 128.949~128.977의
좁은 범위에 모여 있다.

이 밀집도가 설계서 R9의 배경이다. GFS 격자는 간격이 넓어 17기 전부가 같은 격자를
nearest로 갖는다. 그래서 GFS 쪽 `nearest` pooling은 그룹을 구분하지 못하고,
공간 피처의 기여는 주로 LDAPS에서 나온다.

이제 좌표 없이 공간 프리셋을 실행하면 어떻게 되는지 확인한다.

In [11]:
guardChecks = []

# 공간 프리셋인데 터빈 좌표를 주지 않으면 조용히 건너뛰지 않고 실패해야 한다
try:
  build_feature_pipeline(
    get_feature_set("spatial_idw2_all_group"),
    train_time_index=pilotTrainIndex,
    test_time_index=pilotTestIndex,
    train_ldaps=ldapsTrain[monthMask(ldapsTrain, "forecast_kst_dtm", pilotTrainMonth)],
    train_gfs=gfsTrain[monthMask(gfsTrain, "forecast_kst_dtm", pilotTrainMonth)],
    test_ldaps=ldapsTrain[monthMask(ldapsTrain, "forecast_kst_dtm", pilotTestMonth)],
    test_gfs=gfsTrain[monthMask(gfsTrain, "forecast_kst_dtm", pilotTestMonth)],
    turbine_locations=None,
  )
  guardChecks.append(("공간 프리셋 + 좌표 없음", "통과해 버림", "-"))
except ValueError as error:
  guardChecks.append(("공간 프리셋 + 좌표 없음", "차단", str(error)[:70]))

# 비공간 프리셋은 좌표 없이도 정상 동작해야 한다
meanOnly = build_feature_pipeline(
  get_feature_set("official_mean"),
  train_time_index=pilotTrainIndex,
  test_time_index=pilotTestIndex,
  train_ldaps=ldapsTrain[monthMask(ldapsTrain, "forecast_kst_dtm", pilotTrainMonth)],
  train_gfs=gfsTrain[monthMask(gfsTrain, "forecast_kst_dtm", pilotTrainMonth)],
  test_ldaps=ldapsTrain[monthMask(ldapsTrain, "forecast_kst_dtm", pilotTestMonth)],
  test_gfs=gfsTrain[monthMask(gfsTrain, "forecast_kst_dtm", pilotTestMonth)],
  turbine_locations=None,
)
guardChecks.append(("비공간 프리셋 + 좌표 없음", "정상", f"{len(meanOnly.feature_columns)} features"))

pd.DataFrame(guardChecks, columns=["시나리오", "결과", "메시지"])

,시나리오,결과,메시지
0,공간 프리셋 + 좌표 없음,차단,spatial_idw2_all_group은 공간 pooling을 사용하므로 터빈 좌...
1,비공간 프리셋 + 좌표 없음,정상,76 features


### 4-1. 좌표가 없으면 공간 피처를 생략하는 것이 아니라 실행 자체가 멈춘다

설계서 4.2절의 검증 규칙이 그대로 지켜졌다. 조용한 생략은 최악의 실패 모드다.
`--feature-set spatial_idw2_all_group`으로 실행했는데 `--info-xlsx`를 빠뜨렸을 때,
파이프라인이 319개 피처로 조용히 학습하면 사용자는 550개짜리 모델을 얻었다고 믿는다.
metadata에는 `spatial_idw2_all_group`이 각인되어 있으므로 그 기록마저 거짓이 된다.

반면 비공간 프리셋은 좌표 없이도 정상 동작한다. 필요할 때만 요구하는 계약이다.

---

## 5. own-group superset — 한 번 조립하고 target별로 자른다

`spatial_nearest_own_group`은 target마다 다른 컬럼을 쓴다.
그런데 파이프라인은 하나다. 어떻게 화해시켰는가.

### Decision Box ⑨ — 파이프라인을 세 번 돌리지 않고 superset 하나를 만든 뒤 슬라이스한다

**선택지**

- (A) target마다 파이프라인을 따로 돌려 396개짜리 matrix를 세 벌 만든다
- (B) all-group superset 550개를 한 번 만들고, 학습 직전에 target별로 396개를 고른다

**근거**

(A)는 개념적으로 깔끔하지만 조립이 3배가 되고, 무엇보다 **all-group 경로와 코드가 갈린다.**
`scope`에 따라 파이프라인 구조 자체가 달라지면 all-group일 때 승격 전과 같은 동작을
한다는 보장이 사라진다. 설계서 R3가 경계한 지점이다.

(B)는 `scope="all_group"`이면 세 target이 모두 superset 전체를 가리키므로 **기존 동작과
완전히 동일**하고, `own_group`일 때만 슬라이스가 일어난다. bundle에 `target_feature_columns`
하나를 추가하는 것으로 끝난다.

**채택: (B)**

In [12]:
ownGroup = pipelines["spatial_nearest_own_group"]
allGroup = pipelines["spatial_idw2_all_group"]
supersetColumns = list(ownGroup.feature_columns)

sliceRows = []
for target, columns in ownGroup.target_feature_columns.items():
  groupId = target.removeprefix("kpx_")
  pooled = [name for name in columns if POOLED_PATTERN.search(name)]
  ownPooled = [name for name in pooled if f"_{groupId}_" in name]
  sliceRows.append((target, len(columns), len(pooled), len(ownPooled), pooled[0] if pooled else "-"))

sliceFrame = pd.DataFrame(
  sliceRows,
  columns=["target", "사용 컬럼", "pooling 컬럼", "자기 그룹 pooling", "첫 pooling 컬럼"],
)
print(f"superset          : {len(supersetColumns)}")
print(f"all_group target  : {sorted({len(c) for c in allGroup.target_feature_columns.values()})}")
print(f"own_group target  : {sorted({len(c) for c in ownGroup.target_feature_columns.values()})}")
print("target 간 컬럼 집합이 서로 다른가 :",
      len({tuple(c) for c in ownGroup.target_feature_columns.values()}) == 3)
sliceFrame

superset          : 550
all_group target  : [550]
own_group target  : [396]
target 간 컬럼 집합이 서로 다른가 : True


,target,사용 컬럼,pooling 컬럼,자기 그룹 pooling,첫 pooling 컬럼
0,kpx_group_1,396,77,77,ldaps_heightAboveGround_10_10u_group_1_nearest
1,kpx_group_2,396,77,77,ldaps_heightAboveGround_10_10u_group_2_nearest
2,kpx_group_3,396,77,77,ldaps_heightAboveGround_10_10u_group_3_nearest


### 5-1. superset 550에서 각 target이 자기 그룹 77개만 남겨 396이 된다

550 − 231(전체 pooling) + 77(자기 그룹만) = 396이다. 세 target의 컬럼 집합은 서로 다르고,
공통 319개(calendar + 집계)는 모두가 공유한다. all-group 프리셋은 세 target이 같은 550을
가리켜 **승격 전 bundle과 구조가 동일**하다.

여기서 하나가 더 남는다. imputer는 superset 550에 fit하는데, own-group 모델은 396만 쓴다.
전체에 fit한 median과 부분집합에 fit한 median이 다르면 own-group 경로가 오염된다.

### Decision Box ⑩ — imputer는 target별로 나누지 않고 superset 하나에 fit한다

**선택지**

- (A) target마다 396개 컬럼에 imputer를 따로 fit한다
- (B) superset 550에 imputer 하나를 fit하고, 슬라이스는 그 뒤에 한다

**근거**

median은 **컬럼 단위 통계**다. 어떤 컬럼의 median은 그 컬럼의 값들로만 정해지고,
같은 프레임에 다른 컬럼이 몇 개 더 있는지와 무관하다. 따라서 (A)와 (B)는 값이 같아야 한다.
같다면 (B)가 낫다 — imputer가 하나면 bundle 직렬화도, inference 경로도 단순해진다.

다만 "같아야 한다"는 논증이지 증거가 아니다. 설계서 2.4절이 이 등가성을 테스트로 잠그라고
명시한 이유이며, 아래에서 공식 데이터로 직접 확인한다.

**채택: (B)** — 등가성이 성립하는 한에서.

In [13]:
from sklearn.impute import SimpleImputer

trainMatrix = ownGroup.train_matrix
supersetImputer = SimpleImputer(strategy="median").fit(trainMatrix)
supersetMedians = pd.Series(supersetImputer.statistics_, index=trainMatrix.columns)

equivalenceRows = []
for target, columns in ownGroup.target_feature_columns.items():
  subsetImputer = SimpleImputer(strategy="median").fit(trainMatrix[list(columns)])
  subsetMedians = pd.Series(subsetImputer.statistics_, index=list(columns))
  difference = (subsetMedians - supersetMedians.loc[list(columns)]).abs()
  equivalenceRows.append(
    (target, len(columns), float(difference.max()), int((difference > 0).sum()))
  )

equivalenceFrame = pd.DataFrame(
  equivalenceRows,
  columns=["target", "부분집합 컬럼", "median 최대 절대차", "값이 다른 컬럼 수"],
)
print("NaN이 하나라도 있는 train 컬럼 수 :", int(trainMatrix.isna().any().sum()))
print("NaN이 하나라도 있는 test 컬럼 수  :", int(ownGroup.test_matrix.isna().any().sum()))
equivalenceFrame

NaN이 하나라도 있는 train 컬럼 수 : 0
NaN이 하나라도 있는 test 컬럼 수  : 133


,target,부분집합 컬럼,median 최대 절대차,값이 다른 컬럼 수
0,kpx_group_1,396,0.0,0
1,kpx_group_2,396,0.0,0
2,kpx_group_3,396,0.0,0


### 5-2. 부분집합 median은 superset median과 완전히 같다

세 target 모두 최대 절대차 0.0, 값이 다른 컬럼 0개다. median의 컬럼 독립성이 공식 데이터에서
확인됐고, Decision Box ⑩의 전제가 성립한다.

이 등가성은 편의가 아니라 **누수 방지 장치**이기도 하다. imputer가 하나뿐이므로 inference
경로에서 target별로 다른 통계를 실수로 적용할 여지가 없다. 3절의 fit 경계와 같은 원리다 —
값을 채우는 통계는 오직 train feature matrix에서만 나온다.

한편 결측 분포가 train과 test에서 **완전히 갈린다.** train 550개 컬럼에는 NaN이 하나도 없고,
test는 133개 컬럼(24%)에 NaN이 있다. 설계서 R5가 기록한 2025년 LDAPS 결측이며,
이 값들은 train에서 fit한 median으로 채워진다. 즉 2025년 제출용 피처의 4분의 1은
관측이 아니라 **2022~2024년 중앙값**이다. 이 사실은 11번 노트북의 점수 해석에서
반드시 함께 읽어야 한다 — local holdout(2024)에는 없는 결측이 제출 시점(2025)에만
대량으로 존재하므로, local 점수와 Public 점수의 괴리가 생길 수 있는 지점이다.

---

## 6. 종합 결론

### 6-1. 연구 질문

> 피처셋 config가 만드는 조립 순서와 fit 경계가 train/test 대칭을 보장하는가?

**보장한다.** 조립기는 네 단계를 계약된 순서로 실행해 다섯 구획이 연속 블록을 이루는
컬럼 배치를 만들고, 마지막에 train/test 컬럼 동일성을 사후 조건으로 강제한다.
fit은 train grid geometry와 train feature matrix에서만 일어나며, 어긋나면 흡수하지 않고 멈춘다.

### 6-2. 단계별 요약

| 절 | 확인한 것 | 결과 |
|----|-----------|------|
| 1 | 조립 4단계와 순서 계약 | ③의 입력은 ①이 끝난 raw 행 (Decision Box ⑥) |
| 2 | 컬럼 구획과 프리셋별 피처 수 | 9/133/177/99/132 = 550, 랩 노트북과 일치 |
| 2-3 | ①을 끈 대조 실험 | pooling 대상 77 → 65, 최종 550 → 466 |
| 3 | fit 경계와 drift 검출 | 좌표 이동·격자 소멸·스키마 위반 모두 차단 |
| 4 | 터빈 좌표 로더 | 17기 6·6·5, 21.6/21.6/21.0 MW 검증 통과 |
| 5 | own-group 슬라이스와 median | superset 550 → target 396, median 최대차 0.0 |

### 6-3. 주요 발견

1. **①과 ③은 독립 스위치가 아니다.** wind vector를 끄면 공간 pooling 대상에서 12개
   변수가 통째로 사라져 최종 피처가 84개 줄었다. 두 옵션이 곱셈으로 상호작용하므로
   순서를 바꾸면 노트북 08의 550개는 어떤 조합으로도 재현되지 않는다.
2. **조용한 생략이 가장 위험한 실패다.** 좌표 없이 공간 프리셋을 부르면 파이프라인이
   멈춘다. 만약 생략하고 진행했다면 metadata에는 공간 프리셋이 각인된 채 319개짜리
   모델이 만들어져, 재현성 기록 자체가 거짓이 된다.
3. **median의 컬럼 독립성이 own-group 설계를 떠받친다.** 이 성질이 없었다면 target마다
   imputer를 따로 두어야 했고, bundle 스키마와 inference 경로가 3배로 복잡해졌을 것이다.

### 6-4. 시사점

전처리 누수는 대개 "이건 label이 아니니 봐도 된다"는 판단에서 시작한다. 격자 좌표가
정확히 그런 값이다. Moscovich와 Rosset이 보인 대로 비지도 전처리도 resampling 밖에서
추정하면 편향을 만들며, 시계열에서는 그 편향이 look-ahead bias와 겹친다.
이 파이프라인이 택한 방어는 규율이 아니라 **구조**다 — fit 대상을 train으로 고정하고,
불일치를 흡수 대신 예외로 바꿔 두면 사람이 주의하지 않아도 누수가 통과하지 못한다.

### 6-5. 한계

- 이 노트북은 **점수를 재지 않는다.** 조립이 옳다는 것과 그 조립이 랩 점수를 재현한다는
  것은 다른 명제이며, 후자는 11번 노트북이 닫는다.
- drift 검출은 격자 좌표와 컬럼 스키마까지다. 같은 격자에서 **관측 품질이 달라지는**
  drift(예: 2025년 LDAPS 결측 확대)는 검출하지 않고 NaN으로 통과시킨다.
- 파일럿은 2022년 1~2월 한 쌍만 확인했다. 계절별로 격자 구성이 달라지는 경우는
  전체 조립에서만 간접적으로 검증됐다.

### 6-6. 요약

조립기는 네 단계를 계약 순서로 실행해 **train/test 대칭인 feature matrix**를 만들고,
fit 범위를 train으로 가둔 뒤 어긋남을 예외로 드러낸다. own-group은 superset 한 벌에서
슬라이스되며 median 등가성이 이를 뒷받침한다.
다음 노트북(11)은 이렇게 조립한 파이프라인이 노트북 08의 `0.592485`를 재현하는지 확인하고,
첫 제출 후보를 만든다.

---

## 산출물 안전 확인

이 노트북은 모델·제출물·registry를 만들지 않는다. 메모리 안에서만 조립했다.

In [14]:
artifactNames = ("outputs", "submission.csv", "baseline.pkl", "models")
existing = [name for name in artifactNames if (projectRoot / name).exists()]

# 파일럿 1 + 프리셋 4종 + ① 대조군 1 + 좌표 없는 비공간 1
assembledCount = 1 + len(pipelines) + 1 + 1
print("조립한 파이프라인 수 :", assembledCount)
print("메모리 내 superset 최대 :", max(len(p.feature_columns) for p in pipelines.values()))
print("프로젝트 내부 산출물 :", existing if existing else "없음")
print("data/raw/open 원본 수정 여부 :", ldapsTrain.shape == (420864, 35) and gfsTest.shape == (78840, 40))

조립한 파이프라인 수 : 7
메모리 내 superset 최대 : 550
프로젝트 내부 산출물 : 없음
data/raw/open 원본 수정 여부 : True


프로젝트 안에는 아무 산출물도 만들지 않았다. 조립 계약 검증만 수행하고 종료한다.